> **⚠ SUPERSEDED FRAMING (2026-06-28).** This notebook is a point-in-time record (Carroll 2020 6-Parameter Scalar Recovery (Track 1 v0)); its framing predates the surrogate-to-model **identifiability study** that is now canonical (four observable parameters {`alpfe`, `scav_rat`, `diatomgraz`, `R_PICPOC`}; the growth pair {`Smallgrow`, `Biggrow`} is unobservable by construction; `R_PICPOC` is recoverable via a real calcite anchor; per-cell prediction is load-bearing for the {`alpfe`, `scav_rat`, `R_PICPOC`} trio, 7/10 vs 0/10 global-scalar). The 0-D surrogate gap is dimensional (the box homogenizes spatial structure). Treat its numbers and claims as historical. Canonical results: `STATUS.md` / `README.md`.

# ECCO-DarwinDiff — Carroll 2020 6-Parameter Scalar Recovery (Track 1 v0)

Step 5 of the prototype arc, and the first notebook scoped against the post-2026-05-07-call decision: the **exact six parameters Carroll et al. 2020 (JAMES) tuned via Green's functions**.

Notebooks 1–4 worked with depth-resolved profiles ($\mu(z)$, mortality$(z)$, …) on toy reaction-diffusion columns. That validated the differentiable-physics scaffold but did not yet target the parameters the ECCO-Darwin team operationally calibrates. **This notebook does.**

**The six Carroll 2020 parameters (verified against the original Darwin 1 source build at `MITgcm-contrib/ecco_darwin/v04/llc270_JAMES_paper`, see `docs/ecco_darwin_parameter_inventory.md`):**

| # | Source variable (Darwin 1) | Paper Table 1 name | Carroll-optimised value | Plausible search range |
|---|---|---|---|---|
| 1 | `alpfe` | Iron dust solubility | 0.92831 | [0.05, 1.0] |
| 2 | `scav_rat` | Iron scavenging rate (s⁻¹) | 6.03 × 10⁻⁷ | [3 × 10⁻⁸, 3 × 10⁻⁶] |
| 3 | `Smallgrow` | Small phytoplankton growth rate (d⁻¹) | 0.66098 | [0.10, 2.0] |
| 4 | `Biggrow` | Large phytoplankton growth rate (d⁻¹) | 0.43148 | [0.10, 2.0] |
| 5 | `diatomgraz` | Diatom palatability (–) | 0.83003 | [0.05, 1.0] |
| 6 | `R_PICPOC` | PIC/POC ratio (–) | 0.04245 | [0.005, 0.20] |

**Implementation backend: Darwin 3 (ECCO-Darwin v5).** Carroll 2020's paper used **Darwin 1** (hardcoded Fortran in `v04/llc270_JAMES_paper`, where the source-variable names in the table above live). DarwinDiff implements against **Darwin 3** ([github.com/darwinproject/darwin3](https://github.com/darwinproject/darwin3)), the BGC core used by **ECCO-Darwin v5** — confirmed on the 2026-05-07 call. The six parameter *names* and Carroll-optimised *values* are still the scientific test target, but their Darwin 3 namelist mapping is an open task: `alpfe` is still `alpfe` in `&DARWIN_PARAMS`; `scav_rat` likely maps to `scav_tau` (timescale, the inverse); `Smallgrow` and `Biggrow` map to specific entries of the `a_PCmax(np)` array indexed by phytoplankton group; `diatomgraz` maps to specific `PALAT` matrix entries in `data.traits`; `R_PICPOC`'s Darwin 3 home is not yet verified. The 0-D box model in *this* notebook uses Carroll's names directly and is a Darwin-version-agnostic proxy — the Darwin 3 namelist mapping enters in `06`.

**Why a 0-D box model instead of LLC270.** Carroll fit these as global scalars; the immediate test is *can autodiff recover the same six numbers as Green's functions, on the same kind of system, in one training run instead of one-knob-at-a-time?* A scoped 0-D Fe–N-phytoplankton-POC-PIC box model exposes all six knobs with one primary observable each. Once recovery is demonstrated against synthetic obs here, the next notebook swaps in real ECCO-Darwin v5 surface output subset to the **Mid-Atlantic and Pacific AOIs** (chosen on the call for the densest vessel-recorded observations).

**Status: scaffolding notebook.** Real ECCO-Darwin output and ship-cruise / BGC-Argo vessel observations are TODOs marked in the closing section. Synthetic observations from a known-truth parameter vector keep the first pass honest in the same way as notebooks 1–4.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}")
print(
    f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}"
)

## 1. Carroll-6 box model

Surface mixed-layer scalar BGC with five prognostic tracers — dissolved iron (DFe), small phytoplankton ($P_s$), large phytoplankton / diatom ($P_l$), particulate organic carbon (POC), particulate inorganic carbon (PIC) — and constant external dust + light forcing. Every Carroll-2020-tuned parameter has a primary effect on a distinct observable, which is what makes the 6-param inverse problem well-posed without a 3-D model.

$$
\begin{aligned}
\frac{d\,\text{DFe}}{dt} &= \alpha_{\text{Fe}} \cdot \Phi_{\text{dust}} - k_{\text{scav}} \cdot \text{DFe} \cdot \text{POC} - Q_{\text{Fe}}\,(\mu_s P_s + \mu_l P_l)\, f_{\text{Fe}}(\text{DFe}) \\
\frac{dP_s}{dt} &= \mu_s\, f_{\text{Fe}}\, P_s - m_{\text{lin}} P_s - m_{\text{quad}} P_s^{2} \\
\frac{dP_l}{dt} &= \mu_l\, f_{\text{Fe}}\, P_l - m_{\text{lin}} P_l - m_{\text{quad}} P_l^{2} - g_{\text{diatom}}\, G_0\, P_l \\
\frac{d\,\text{POC}}{dt} &= M_{\text{tot}} - w_{\text{sink}} \cdot \text{POC} \\
\frac{d\,\text{PIC}}{dt} &= R_{\text{PIC/POC}} \cdot M_{\text{tot}} - w_{\text{sink}} \cdot \text{PIC}
\end{aligned}
$$

where $f_{\text{Fe}}(\text{DFe}) = \text{DFe} / (\text{DFe} + K_{\text{Fe}})$ and $M_{\text{tot}}$ is the sum of all phyto loss fluxes (mortality + diatom grazing).

**Carroll knob → primary observable:**
- $\alpha_{\text{Fe}} = $ `alpfe` → steady-state DFe set by source term.
- $k_{\text{scav}} = $ `scav_rat` → sensitivity of DFe to POC (breaks the alpfe degeneracy).
- $\mu_s = $ `Smallgrow` → steady-state $P_s$.
- $\mu_l = $ `Biggrow` → steady-state $P_l$ (modulated by grazing).
- $g_{\text{diatom}} = $ `diatomgraz` → grazing loss on $P_l$ (separates `Biggrow` from `Smallgrow` signal).
- $R_{\text{PIC/POC}} = $ `R_PICPOC` → PIC magnitude relative to POC.

Background defaults ($m_{\text{lin}}$, $m_{\text{quad}}$, $G_0$, $w_{\text{sink}}$, $K_{\text{Fe}}$, $\Phi_{\text{dust}}$, $Q_{\text{Fe}}$) are fixed at literature-plausible values and **not learned** — they correspond to the ~94 % of Darwin parameters Carroll left at expert defaults.

In [ ]:
# Carroll-6 box model. Inline while equations are still mutable; will move to
# src/darwindiff/carroll6.py once the model is stable.

# Background (non-learned) parameters — the ~94% of Darwin knobs Carroll left at defaults.
M_LIN = 0.05         # 1/d, linear phyto mortality
M_QUAD = 0.50        # 1/d / (mmol C/m^3), quadratic phyto mortality
G0_GRAZE = 0.30      # 1/d, baseline grazing rate (multiplied by diatomgraz to give Pl loss)
W_SINK = 0.10        # 1/d, particulate sinking rate
K_FE = 5.0e-5        # mmol Fe/m^3, iron half-saturation for phyto growth
PHI_DUST = 5.0e-5    # mmol Fe/m^3/d, dissolved-iron source from dust
Q_FE = 1.0e-5        # mol Fe / mol C, phyto Fe quota
LIGHT = 1.0          # surface light, dimensionless


def carroll6_step(
    state: torch.Tensor,   # shape [5]: [DFe, Ps, Pl, POC, PIC]
    params: torch.Tensor,  # shape [6]: [alpfe, scav_rat, Smallgrow, Biggrow, diatomgraz, R_PICPOC]
    dt: float,             # in days
) -> torch.Tensor:
    DFe, Ps, Pl, POC, PIC = state[0], state[1], state[2], state[3], state[4]
    alpfe, scav_rat, mu_s, mu_l, g_diatom, R_PICPOC = (
        params[0], params[1], params[2], params[3], params[4], params[5]
    )
    # scav_rat is per-second in the Carroll source; convert at use-site.
    scav_rat_per_day = scav_rat * 86400.0

    f_fe = DFe / (DFe + K_FE)
    growth_s = mu_s * f_fe * LIGHT * Ps
    growth_l = mu_l * f_fe * LIGHT * Pl
    fe_uptake = Q_FE * (growth_s + growth_l)
    mort_s = M_LIN * Ps + M_QUAD * Ps * Ps
    mort_l = M_LIN * Pl + M_QUAD * Pl * Pl
    graze_l = g_diatom * G0_GRAZE * Pl
    mort_total = mort_s + mort_l + graze_l

    dDFe = alpfe * PHI_DUST - scav_rat_per_day * DFe * POC - fe_uptake
    dPs = growth_s - mort_s
    dPl = growth_l - mort_l - graze_l
    dPOC = mort_total - W_SINK * POC
    dPIC = R_PICPOC * mort_total - W_SINK * PIC

    return torch.stack([
        DFe + dt * dDFe,
        Ps  + dt * dPs,
        Pl  + dt * dPl,
        POC + dt * dPOC,
        PIC + dt * dPIC,
    ])


def carroll6_integrate(
    state0: torch.Tensor,
    params: torch.Tensor,
    dt: float,
    n_steps: int,
    snapshot_indices: list[int] | None = None,
) -> torch.Tensor:
    """Forward-Euler integration with autograd through every step.

    Returns final state if snapshot_indices is None, or shape [n_snap, 5] otherwise.
    """
    state = state0
    if snapshot_indices is None:
        for _ in range(n_steps):
            state = carroll6_step(state, params, dt)
        return state
    snapshot_set = set(snapshot_indices)
    snaps: list[torch.Tensor] = []
    for step in range(1, n_steps + 1):
        state = carroll6_step(state, params, dt)
        if step in snapshot_set:
            snaps.append(state)
    return torch.stack(snaps)


# Carroll 2020 ground truth: (alpfe, scav_rat, Smallgrow, Biggrow, diatomgraz, R_PICPOC).
CARROLL_VALUES = torch.tensor([
    0.92831,
    10.41124 * 0.005 / 86400.0,  # = 6.026e-7 per second
    0.66098,
    0.43148,
    0.83003,
    0.04245,
])
PARAM_NAMES = ["alpfe", "scav_rat", "Smallgrow", "Biggrow", "diatomgraz", "R_PICPOC"]
PARAM_BOUNDS = torch.tensor([
    [0.05, 1.0],     # alpfe
    [3e-8, 3e-6],    # scav_rat (per second), 100x window around Carroll's 6e-7
    [0.10, 2.0],     # Smallgrow (1/d)
    [0.10, 2.0],     # Biggrow (1/d)
    [0.05, 1.0],     # diatomgraz
    [0.005, 0.20],   # R_PICPOC
])

print("Carroll 2020 ground truth (six parameters):")
for n, v in zip(PARAM_NAMES, CARROLL_VALUES):
    print(f"  {n:<11s} = {v.item():.6e}")

## 2. Synthetic observations from the Carroll-2020 truth

Run the box model with the six known-truth values from Carroll's Green's-functions calibration. Take five evenly-spaced snapshots through a 50-day spin-up — short enough that the iron pool is still in transient (which breaks the alpfe / scav_rat product degeneracy a fully-equilibrated run would exhibit), long enough that the phyto pools settle. Add Gaussian noise sized to a plausible vessel-recorded measurement standard deviation (1 % of mean state).

**TODO when real data lands:** swap this cell for *load ECCO-Darwin v5 LLC270 surface output → subset to Mid-Atlantic and Pacific AOI masks → area-average each AOI → drop into the same five-snapshot tensor.* Then replace `state_obs` with the corresponding vessel data composite (GEOTRACES iron, GO-SHIP nitrate / DIC / ALK, BGC-Argo POC proxies).

In [ ]:
state0 = torch.tensor([5.0e-4, 1.0, 1.0, 0.5, 0.025])  # DFe, Ps, Pl, POC, PIC
dt = 0.25            # days
n_steps = 200        # 50-day spin-up
snapshot_indices = [40, 80, 120, 160, 200]

with torch.no_grad():
    state_traj_truth = carroll6_integrate(
        state0=state0,
        params=CARROLL_VALUES,
        dt=dt,
        n_steps=n_steps,
        snapshot_indices=snapshot_indices,
    )

tracer_names = ["DFe", "Ps", "Pl", "POC", "PIC"]
print(f"State trajectory (truth) shape: {tuple(state_traj_truth.shape)}")
print(f"Final state at t = {n_steps * dt:.0f} d:")
for n, v in zip(tracer_names, state_traj_truth[-1]):
    print(f"  {n:<5s} = {v.item():.4e}")

# 1% per-tracer noise stand-in for vessel-recorded observation error.
torch.manual_seed(42)
tracer_means = state_traj_truth.mean(dim=0).clamp(min=1e-12)
noise_std = 0.01 * tracer_means
state_obs = state_traj_truth + noise_std * torch.randn_like(state_traj_truth)
print(f"\nObserved snapshots shape: {tuple(state_obs.shape)}  (5 snapshots x 5 tracers)")

## 3. Six-parameter recovery via autograd

The six unknowns are bounded scalars, learned as $\theta \in \mathbb{R}^{6}$ and mapped through a sigmoid to the physical ranges in the table above. This is the smallest viable parameter-learner — no MLP, no environmental covariates — because the Carroll target itself is six global scalars. The MLP-predicts-spatial-field machinery from notebooks 1–4 enters in **the next** notebook when each scalar is extended to a Mid-Atl/Pacific 2-D field.

Loss is MSE on every observed snapshot of every tracer, normalised per-tracer by the mean truth state so each tracer contributes comparably regardless of unit scale. This is the differentiable analogue of the Carroll Green's-functions cost — same misfit, but optimised with backprop instead of one-knob-at-a-time perturbations.

In [ ]:
def bounded_params(theta: torch.Tensor, bounds: torch.Tensor) -> torch.Tensor:
    """Map unconstrained theta to physical ranges via sigmoid."""
    lo, hi = bounds[:, 0], bounds[:, 1]
    return lo + (hi - lo) * torch.sigmoid(theta)


# Use the auto-detected device from cell 1 (RTX 5090 Laptop GPU when available).
# Note: this 0-D scaffold has only 5-element state tensors, so per-step Python
# dispatch dominates over kernel work and GPU and CPU are similar in wall-time.
# The GPU advantage becomes real in `06+` once spatial dimensions enter the state.
fit_device = device
print(f"Fit device: {fit_device}")

theta = torch.zeros(6, requires_grad=True, device=fit_device)  # sigmoid(0) = 0.5 = midpoint
optimizer = torch.optim.Adam([theta], lr=5e-2)

state_obs_dev = state_obs.to(fit_device)
state0_dev = state0.to(fit_device)
bounds_dev = PARAM_BOUNDS.to(fit_device)
norm = state_traj_truth.mean(dim=0).to(fit_device).clamp(min=1e-12)

n_epochs = 2000
losses: list[float] = []

t0 = time.time()
for epoch in range(n_epochs):
    optimizer.zero_grad()
    params_pred = bounded_params(theta, bounds_dev)
    state_pred = carroll6_integrate(
        state0=state0_dev,
        params=params_pred,
        dt=dt,
        n_steps=n_steps,
        snapshot_indices=snapshot_indices,
    )
    residual = (state_pred - state_obs_dev) / norm
    loss = (residual ** 2).mean()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch + 1:4d}  loss = {loss.item():.4e}")
elapsed = time.time() - t0
print(f"\nFit complete: {n_epochs} epochs in {elapsed:.1f}s on {fit_device}")

## 4. Recovery results vs the Carroll-2020 ground truth

Per-parameter relative error against the published Green's-functions optimum is the basic identifiability score; the loss curve shows convergence.

In [ ]:
with torch.no_grad():
    final_params = bounded_params(theta, bounds_dev).cpu()

print(f"{'Parameter':<12s} {'Truth':>14s} {'Recovered':>14s} {'Rel. error':>12s}")
print("-" * 58)
for name, tru, rec in zip(PARAM_NAMES, CARROLL_VALUES, final_params):
    rel = abs((rec.item() - tru.item()) / tru.item()) * 100
    print(f"{name:<12s} {tru.item():>14.6e} {rec.item():>14.6e} {rel:>10.2f}%")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].semilogy(losses)
ax[0].set_xlabel("epoch")
ax[0].set_ylabel("normalised MSE loss")
ax[0].set_title("Loss curve")
ax[0].grid(alpha=0.3)

x = np.arange(6)
truth = CARROLL_VALUES.numpy()
recov = final_params.numpy()
ax[1].bar(x - 0.2, truth / truth, width=0.4, label="truth (= 1)", color="black", alpha=0.6)
ax[1].bar(x + 0.2, recov / truth, width=0.4, label="recovered / truth", color="tab:green")
ax[1].set_xticks(x)
ax[1].set_xticklabels(PARAM_NAMES, rotation=30, ha="right")
ax[1].set_ylabel("ratio to truth")
ax[1].set_title("Per-parameter recovery")
ax[1].axhline(1.0, color="black", linewidth=0.5, alpha=0.5)
ax[1].legend()
ax[1].grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## What this scaffold demonstrates

**Methodological result:** end-to-end autodiff through a Darwin-shaped reaction network recovers the **exact six Carroll-2020 Green's-functions parameters** within single-digit %, in one training run, against synthetic observations from the published optimum:

| Param | Truth (Carroll 2020) | Recovered | Rel. err |
|---|---|---|---|
| `R_PICPOC` | 4.245 × 10⁻² | 4.224 × 10⁻² | **0.49 %** |
| `Smallgrow` | 0.6610 | 0.6618 | **0.13 %** |
| `Biggrow` | 0.4315 | 0.4176 | **3.22 %** |
| `diatomgraz` | 0.8300 | 0.7941 | **4.32 %** |
| `scav_rat` | 6.025 × 10⁻⁷ | 5.644 × 10⁻⁷ | **6.32 %** |
| `alpfe` | 0.9283 | 0.8594 | **7.43 %** |

Loss converged from ~5.9 to **9.6 × 10⁻⁵** over 2000 Adam epochs (lr=5e-2), at the 1 %-noise floor. Autograd flows through 200 forward-Euler steps × 5 coupled tracers; the optimisation is single-shot on all six knobs simultaneously, the structural difference from the one-knob-at-a-time Green's-functions workflow.

**The methodological-finding worth surfacing** (and the reason the original 200-day spin-up underperformed the 50-day one): **observation timing matters more than epoch count.** Snapshots taken while the iron pool is still equilibrating contain transient information that breaks the alpfe / scav_rat product degeneracy a fully-equilibrated trajectory would exhibit. Same lesson as notebook 4's snapshot-vs-trajectory finding, now demonstrated on the Carroll-2020 target: identifiability is per-parameter and per-observation-type, not a global property of the model.

**Compute observation (relevant to the cluster compute proposal):** at this 0-D scale, the autograd graph has 5-element state tensors, and per-step Python dispatch dominates over kernel work. The 5090 wall time (589.6 s) was ~3.5× slower than CPU (164.9 s) because GPU launch overhead doesn't amortise. This is exactly the regime where GPU parallelism does not help — and exactly why scaling to LLC270 spatial dimensions (millions of cells per timestep, batched coupled tracers) is the workload class that *does* benefit from B200-class GPUs and fast interconnects.

## Does not yet demonstrate

That recovery holds against real ECCO-Darwin v5 (Darwin 3) output or against ship-cruise / BGC-Argo observations. The 0-D box model is a Darwin-version-agnostic proxy; the full LLC270 v5 system has spatial transport, sub-mesoscale variability, and cross-tracer couplings the proxy ignores — and uses the Darwin 3 parameter representation, not Carroll's Darwin 1 source-variable names.

## Planned follow-ups

1. `06_carroll6_real_ecco_darwin.ipynb` — swap the synthetic ground-truth snapshots for actual **ECCO-Darwin v5 (Darwin 3 backend)** surface output, subset to the **Mid-Atlantic and Pacific AOI masks** decided on the 2026-05-07 call (chosen for vessel-data density). First sub-step is the **Carroll-6 → Darwin 3 namelist mapping** (which `data.darwin` / `data.traits` entries correspond to each of `alpfe`, `scav_rat`, `Smallgrow`, `Biggrow`, `diatomgraz`, `R_PICPOC`). Then replicate Carroll's six-parameter fit on each AOI separately and on the joint loss; check whether autodiff with a literature-default prior matches the published Carroll values.
2. `07_carroll6_to_2d_fields.ipynb` — extend each of the six scalars to a 2-D spatial field via a small CNN over (SST, MLD, surface PAR, dust flux, etc.). The structural extension that differentiates DarwinDiff from Green's functions: per-pixel parameter values where Carroll has one number for the entire ocean. **This is the first notebook that actually engages B200-class compute.**

## Open scope items (verify before relying)

- **Carroll-6 → Darwin 3 namelist mapping.** Provisional in `reference_darwin3.md` memory — `alpfe` still `alpfe`; `scav_rat` likely → `scav_tau` (verify direction); `Smallgrow`/`Biggrow` → specific `a_PCmax(np)` indices; `diatomgraz` → `PALAT(prey, predator)` entries; `R_PICPOC` Darwin 3 location unverified. Five-minute task for whoever has the darwin3 repo cloned; otherwise grep [darwinproject/darwin3](https://github.com/darwinproject/darwin3) and the v5 namelists.
- **Vessel observation source for the loss term.** Likely composite of GEOTRACES (iron), GO-SHIP repeat hydrography (nutrients, DIC, ALK), BGC-Argo (chlorophyll, POC proxies). Need to confirm which products the collaboration already curates for the Mid-Atl + Pacific AOIs.
- **ECCO-Darwin v5 LLC270 (Darwin 3) 1992–2024 download path.** Not yet retrieved as of this notebook's commit; this is the blocker for `06`.
- **Compute setup** (the third open question from the call). The 0-D scaffold is tiny enough that CPU outpaces GPU here; `06` (LLC270 v5 subset to two AOIs) likely engages the 5090 once spatial dims appear; `07` (CNN-over-2D + LLC270 gradient passes) is the workload that justifies ORCD B200s.